# Clef 模型结构探索

本 Notebook 研究 **Cloudflare/clef（27B）**，打印官方实现中的完整 `ClefModel`：多模态骨干 `language_model` 和联合决策头 `head`。

默认只下载配置和官方 Python 源码，在 PyTorch **meta device** 上创建结构，不下载权重，不分配完整参数内存，无需 GPU。meta 参数只有形状、类型等信息，**不能用于推理**。

Clef 根据 state 和问题 schema，为每个问题的候选项输出 logits；对每个问题单独 softmax 得到概率。

资料：[官方模型卡](https://huggingface.co/Cloudflare/clef)、[配置](https://huggingface.co/Cloudflare/clef/blob/main/config.json)、[决策头源码](https://huggingface.co/Cloudflare/clef/blob/main/joint_schema_model.py)。下方固定模型仓库 revision，便于复现。


## 1. 准备独立环境

仓库的基础环境固定了 Transformers 4.57.6，不支持本模型的架构。建议在项目根目录建立独立环境；不需要修改原有学习环境：

```bash
python3.13 -m venv .venv-clef
source .venv-clef/bin/activate
python -m pip install -r 08_Clef模型研究/requirements.txt
python -m ipykernel install --user --name clef-study --display-name "Python (clef-study)"
python -m jupyterlab
```

打开本文件，选择 **Python (clef-study)** 内核，然后从上往下运行。首次运行需要访问 Hugging Face。会导入该模型仓库的官方源码，固定版本链接见下一节。


In [1]:
import importlib.util
import json
import sys
from collections import Counter
from pathlib import Path

import torch
import transformers
from accelerate import init_empty_weights
from huggingface_hub import snapshot_download
from transformers import AutoConfig, Qwen3_5ForConditionalGeneration

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


/private/tmp/clef-notebook-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /private/tmp/clef-notebook-venv/lib/python3.13/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /private/tmp/clef-notebook-venv/lib/python3.13/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
Python: 3.13.18
PyTorch: 2.11.0
Transformers: 5.10.2


## 2. 下载配置与源码

只允许下载以下三个文件，排除 `.safetensors` 权重和 tokenizer。`config.json` 描述骨干，`joint_head_config.json` 描述决策头；仅用 `AutoModel` 创建骨干会遗漏 Clef 的决策头。


In [2]:
MODEL_ID = "Cloudflare/clef"
REVISION = "171287b2571042abec9e7b99752aee58216b0ed0"

model_path = Path(snapshot_download(
    repo_id=MODEL_ID,
    revision=REVISION,
    allow_patterns=["config.json", "joint_head_config.json", "joint_schema_model.py"],
))
print("模型:", MODEL_ID)
print("Revision:", REVISION)
print("源码:", f"https://huggingface.co/{MODEL_ID}/blob/{REVISION}/joint_schema_model.py")

config = AutoConfig.from_pretrained(model_path, local_files_only=True)
head_config = json.loads((model_path / "joint_head_config.json").read_text())
print("\n骨干架构:", config.architectures)
print("文本配置:")
print(json.dumps(config.text_config.to_dict(), indent=2, ensure_ascii=False))
print("\n视觉配置:")
print(json.dumps(config.vision_config.to_dict(), indent=2, ensure_ascii=False))
print("\n联合决策头配置:")
print(json.dumps(head_config, indent=2, ensure_ascii=False))


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 3 files:  33%|███▎      | 1/3 [00:02<00:05,  2.75s/it]

Fetching 3 files: 100%|██████████| 3/3 [00:04<00:00,  1.54s/it]

Fetching 3 files: 100%|██████████| 3/3 [00:04<00:00,  1.66s/it]

模型: Cloudflare/clef
Revision: 171287b2571042abec9e7b99752aee58216b0ed0
源码: https://huggingface.co/Cloudflare/clef/blob/171287b2571042abec9e7b99752aee58216b0ed0/joint_schema_model.py

骨干架构: ['Qwen3_5ForConditionalGeneration']
文本配置:
{
  "transformers_version": "5.10.2",
  "architectures": null,
  "output_hidden_states": false,
  "return_dict": true,
  "dtype": "bfloat16",
  "chunk_size_feed_forward": 0,
  "is_encoder_decoder": false,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1"
  },
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1
  },
  "problem_type": null,
  "vocab_size": 248320,
  "hidden_size": 5120,
  "intermediate_size": 17408,
  "num_hidden_layers": 64,
  "num_attention_heads": 24,
  "num_key_value_heads": 4,
  "hidden_act": "silu",
  "max_position_embeddings": 262144,
  "initializer_range": 0.02,
  "rms_norm_eps": 1e-06,
  "use_cache": true,
  "tie_word_embeddings": false,
  "rope_parameters": {
    "mrope_interleaved": true,
    "mrope_section": [
      11,
      11

## 3. 创建并打印完整模型

通过固定路径导入官方 `ClefModel` 和 `JointSchemaHead`。`init_empty_weights` 让构造过程只创建 meta 参数；这里没有加载训练后的权重。


In [3]:
spec = importlib.util.spec_from_file_location(
    "clef_release_model", model_path / "joint_schema_model.py"
)
clef_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = clef_module  # dataclass 需要模块已注册
spec.loader.exec_module(clef_module)

with init_empty_weights(include_buffers=True):
    backbone = Qwen3_5ForConditionalGeneration(config)
    head = clef_module.JointSchemaHead(**head_config)
    model = clef_module.ClefModel(backbone, head)
model.eval()

assert all(p.is_meta for p in model.parameters()), "结构探索不应分配真实参数"
print(model)


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


ClefModel(
  (language_model): Qwen3_5ForConditionalGeneration(
    (model): Qwen3_5Model(
      (visual): Qwen3_5VisionModel(
        (patch_embed): Qwen3_5VisionPatchEmbed(
          (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
        )
        (pos_embed): Embedding(2304, 1152)
        (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
        (blocks): ModuleList(
          (0-26): 27 x Qwen3_5VisionBlock(
            (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (attn): Qwen3_5VisionAttention(
              (qkv): Linear(in_features=1152, out_features=3456, bias=True)
              (proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (mlp): Qwen3_5VisionMLP(
              (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
              (linear_fc2): Linear(in_features=4304, out_features=1152, bias=Tru

## 4. 参数量与关键层

`print(model)` 会折叠连续的相同层。下面另外列出层数、注意力类型、组件参数量和若干参数形状。参数量按当前实例的参数统计；内存估计只包括参数，不包含激活、缓存和运行时开销。


In [4]:
print("文本层数:", config.text_config.num_hidden_layers)
print("文本隐藏维度:", config.text_config.hidden_size)
print("文本注意力层类型:", dict(Counter(config.text_config.layer_types)))
print("视觉层数:", config.vision_config.depth)
print("决策头宽度:", head_config["width"])

for name, component in [("骨干", model.language_model), ("决策头", model.head), ("完整 Clef", model)]:
    count = sum(p.numel() for p in component.parameters())
    print(f"{name}: {count:,} 参数 | 仅 BF16 参数约 {count * 2 / 1024**3:.2f} GiB")

print("\n前 12 个参数的形状（没有真实权重数值）:")
for index, (name, parameter) in enumerate(model.named_parameters()):
    if index >= 12:
        break
    print(f"{name}: shape={tuple(parameter.shape)}, device={parameter.device}")

print("\n独立决策头:")
print(model.head)


文本层数: 64
文本隐藏维度: 5120
文本注意力层类型: {'linear_attention': 48, 'full_attention': 16}
视觉层数: 27
决策头宽度: 1024
骨干: 27,356,728,560 参数 | 仅 BF16 参数约 50.96 GiB
决策头: 128,056,324 参数 | 仅 BF16 参数约 0.24 GiB
完整 Clef: 27,484,784,884 参数 | 仅 BF16 参数约 51.19 GiB

前 12 个参数的形状（没有真实权重数值）:
language_model.model.visual.patch_embed.proj.weight: shape=(1152, 3, 2, 16, 16), device=meta
language_model.model.visual.patch_embed.proj.bias: shape=(1152,), device=meta
language_model.model.visual.pos_embed.weight: shape=(2304, 1152), device=meta
language_model.model.visual.blocks.0.norm1.weight: shape=(1152,), device=meta
language_model.model.visual.blocks.0.norm1.bias: shape=(1152,), device=meta
language_model.model.visual.blocks.0.norm2.weight: shape=(1152,), device=meta
language_model.model.visual.blocks.0.norm2.bias: shape=(1152,), device=meta
language_model.model.visual.blocks.0.attn.qkv.weight: shape=(3456, 1152), device=meta
language_model.model.visual.blocks.0.attn.qkv.bias: shape=(3456,), device=meta
language_model.mo

## 5. 可选：加载真实权重

查看结构到上一节即可。下方默认关闭；仅在有足够显存的 CUDA 环境中，才将开关改为 `True`。这会下载完整的 27B 模型及决策头权重，规模为几十 GB。官方加载函数不提供此处的自动 CPU / 磁盘卸载策略。

加载后 `trained_model` 可用于官方模型卡中的 `encode_record` / `collate_records` / `systemone` 推理流程。上面的 meta `model` 仍然只是结构实例。


In [5]:
LOAD_PRETRAINED = False

if LOAD_PRETRAINED:
    if not torch.cuda.is_available():
        raise RuntimeError("此可选示例要求 CUDA GPU；只查看结构无需开启。")
    release_path = snapshot_download(repo_id=MODEL_ID, revision=REVISION)
    trained_model, processor = clef_module.load_release_model(
        release_path, device="cuda", dtype=torch.bfloat16
    )
    print(trained_model)
else:
    print("已跳过真实权重下载；上方已打印完整 Clef 模型结构。")


已跳过真实权重下载；上方已打印完整 Clef 模型结构。
